In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, to_date
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

# Create schema if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS novocart.bronze")

# Tables with their primary keys
table_config = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "exchange_rates": "currency_code"
}

for tbl, pk in table_config.items():
    
    print(f"Processing table: {tbl}")
    
    # Read from source (table)
    df = spark.table(f"azure_blob_storage.{tbl}")
    
    # Special handling for exchange_rates: convert effective_date to DATE type
    if tbl == "exchange_rates":
        df = df.withColumn("effective_date", to_date(df["effective_date"], "dd/MM/yyyy"))
    
    # Add audit column (UC supported)
    df = df.withColumn("ingestion_time", current_timestamp())
    
    target_table = f"novocart.bronze.{tbl}_raw"
    
    # Check if table exists
    if spark.catalog.tableExists(target_table):
        
        delta_table = DeltaTable.forName(spark, target_table)
        
        merge_condition = f"tgt.{pk} = src.{pk}"
        
        # Incremental load (only new records)
        delta_table.alias("tgt").merge(
            df.alias("src"),
            merge_condition
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        
        print(f"Incremental load completed for {tbl}")
    
    else:
        # First time load
        df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)
        
        print(f"Initial load completed for {tbl}")